#  Сквозной математический конвейер и генерация эталонных кейсов
**Задачи 2–6: Расчет равновесной ставки $r^*$, вероятности кассового разрыва $P_{\text{gap}}$ и формирование 10 персон**

1. **Фактор долговой нагрузки (`debt_load`):** отношение текущих платежей $D_{\text{old}}$ к операционному денежному потоку $\mu = \text{Revenue} \times \text{margin}$.
2. **Вероятность дефолта (`pd_client`):** расчет по баллам из `scorecard_bins.csv` через параметры калибровки (`PDO = 20`, `Base Score = 600`, `Base Odds = 50`).
3. **Равновесная ставка (`rate` / $r^*$):** выводится из ключевой ставки ЦБ РФ, маржи продукта и премии за риск дефолта: $r^* = \frac{r_0 + \text{PD}}{1 - \text{PD}}$.
4. **Вероятность кассового разрыва (`p_gap`):** моделируется через нормальное распределение Гаусса ($\Phi(Z)$ с использованием `math.erf`) и учитывает финансовую подушку остатка на счете клиента $C_0$.
5. **Точки А и Б (`status`, `offer_amount`, `offer_term`):** автоматическая классификация на 3 зоны (Зеленая, Желтая, Красная) и реструктуризация параметров кредита при высоком риске кассового разрыва.

In [13]:
import pandas as pd
import math
from IPython.display import display

# Настройка формата вывода чисел в таблицах
pd.options.display.float_format = '{:,.4f}'.format

# 1. Загружаем скоринговую карту и параметры калибровки
df_bins = pd.read_csv('scorecard_bins.csv')
df_params = pd.read_csv('scorecard_params.csv')

BASE_SCORE = int(df_params['base_score'].iloc[0]) # 600
BASE_ODDS = float(df_params['base_odds'].iloc[0]) # 50
PDO = float(df_params['pdo'].iloc[0])             # 20

FACTOR = PDO / math.log(2.0)
OFFSET = BASE_SCORE - (FACTOR * math.log(BASE_ODDS))

# 2. Формируем и сохраняем справочник отраслей (industry_params.csv)
industry_data = [
    {'okved_group': 'it',            'margin': 0.35, 'revenue_cv': 0.15},
    {'okved_group': 'retail',        'margin': 0.15, 'revenue_cv': 0.10},
    {'okved_group': 'construction',  'margin': 0.20, 'revenue_cv': 0.25},
    {'okved_group': 'manufacturing', 'margin': 0.25, 'revenue_cv': 0.12},
    {'okved_group': 'horeca',        'margin': 0.18, 'revenue_cv': 0.22}
]
df_industry = pd.DataFrame(industry_data)
df_industry.to_csv('industry_params.csv', index=False)
ind_map = df_industry.set_index('okved_group').to_dict('index')

# 3. Константы ставок: Ключевая ставка ЦБ РФ + базовая маржа кредитного продукта
KEY_RATE = 0.16
PRODUCT_MARGIN = {
    'WORKING_CAP': 0.03, # Оборотный кредит (r_0 = 0.19)
    'EQUIPMENT':   0.02, # Кредит на оборудование (r_0 = 0.18)
    'CASH_GAP':    0.04  # На покрытие кассового разрыва (r_0 = 0.20)
}

print("✅ Справочники и параметры калибровки загружены!")
display(df_industry)

✅ Справочники и параметры калибровки загружены!


,okved_group,margin,revenue_cv
0,it,0.3500,0.1500
1,retail,0.1500,0.1000
2,construction,0.2000,0.2500
3,manufacturing,0.2500,0.1200
4,horeca,0.1800,0.2200


### Функции расчета

* Порог **Точки А** (Зеленая зона): `PD_client < 0.05` и `P_gap < 0.25`.
* Порог **Точки Б** (Красная зона / стоп-фактор): `PD_client >= 0.15`.
* В **Желтой зоне** (`APPROVED_WITH_CHANGES`) срок кредита увеличивается на 12 месяцев, а ежемесячный платеж ограничивается уровнем 70% от свободного денежного потока клиента.

In [14]:
def get_points(factor_name, val):
    """Поиск баллов по таблице scorecard_bins.csv"""
    sub = df_bins[df_bins['factor'] == factor_name]
    for _, row in sub.iterrows():
        if pd.notna(row['category']):
            if str(val).lower() == str(row['category']).lower():
                return int(row['points'])
        else:
            low = float('-inf') if pd.isna(row['lower']) else float(row['lower'])
            up = float('inf') if pd.isna(row['upper']) else float(row['upper'])
            if low < float(val) <= up:
                return int(row['points'])
    return 0

def annuity_factor(r_m, n):
    """Коэффициент ежемесячного аннуитетного платежа"""
    return (r_m * ((1.0 + r_m) ** n)) / (((1.0 + r_m) ** n) - 1.0)

def solve_persona(p):
    """Полный расчет кредитного решения для одной компании (персоны)"""
    ind = ind_map[p['okved_group']]
    margin, cv = ind['margin'], ind['revenue_cv']

    # 1. Операционный поток (mu) и коэффициент долговой нагрузки (debt_load = D_old / mu)
    mu = p['monthly_revenue'] * margin
    sigma = max(mu * cv, 1.0)
    debt_load = round(p['d_old'] / mu, 4)

    # 2. Суммирование баллов и расчет PD_client (Задача 1)
    score = BASE_SCORE
    score += get_points('business_age_months', p['business_age_months'])
    score += get_points('debt_burden_ratio', debt_load)
    score += get_points('business_type', p['business_type'])
    score += get_points('has_delinquency', p['has_delinquency'])

    odds = math.exp((score - OFFSET) / FACTOR)
    pd_client = round(1.0 / (1.0 + odds), 4)

    # 3. Проверка на Красную зону (Точка Б: PD_client >= 0.15)
    if pd_client >= 0.15:
        return {**p, 'debt_burden_ratio': debt_load, 'score': score, 'pd_client': pd_client,
                'p_gap': 1.0000, 'rate': 0.0, 'payment': 0.0, 'status': 'REJECTED',
                'offer_amount': 0.0, 'offer_term': 0}

    # 4. Базовая ставка r_0 и равновесная ставка r* (Задача 2)
    r_0 = KEY_RATE + PRODUCT_MARGIN[p['product_code']]
    r_star = round((r_0 + pd_client) / (1.0 - pd_client), 4)
    r_m = r_star / 12.0

    payment = round(p['amount'] * annuity_factor(r_m, p['term_months']), 2)

    # 5. Расчет P_gap с учетом подушки остатка на счете C_0 через функцию Гаусса erf (Задача 3)
    available_cash = mu - p['d_old'] + (p['c_0'] / p['term_months'])
    z = (payment - available_cash) / sigma
    p_gap = round(0.5 * (1.0 + math.erf(z / math.sqrt(2.0))), 4)

    # 6. Классификация по Точкам А и Б и расчет альтернативного оффера (Задача 4)
    if pd_client < 0.05 and p_gap < 0.25:
        return {**p, 'debt_burden_ratio': debt_load, 'score': score, 'pd_client': pd_client,
                'p_gap': p_gap, 'rate': r_star, 'payment': payment, 'status': 'APPROVED',
                'offer_amount': float(p['amount']), 'offer_term': int(p['term_months'])}
    else:
        offer_term = min(p['term_months'] + 12, 60)
        avail_new = mu - p['d_old'] + (p['c_0'] / offer_term)
        safe_payment = 0.70 * avail_new

        new_ann = annuity_factor(r_m, offer_term)
        max_safe_amount = round(safe_payment / new_ann, -4)
        offer_amount = float(min(p['amount'], max_safe_amount))

        # Если доступный лимит меньше минимального порога 100 тыс. руб. -> Отказ
        if offer_amount < 100_000:
            return {**p, 'debt_burden_ratio': debt_load, 'score': score, 'pd_client': pd_client,
                    'p_gap': p_gap, 'rate': 0.0, 'payment': 0.0, 'status': 'REJECTED',
                    'offer_amount': 0.0, 'offer_term': 0}

        new_payment = round(offer_amount * new_ann, 2)
        return {**p, 'debt_burden_ratio': debt_load, 'score': score, 'pd_client': pd_client,
                'p_gap': p_gap, 'rate': r_star, 'payment': new_payment,
                'status': 'APPROVED_WITH_CHANGES', 'offer_amount': offer_amount, 'offer_term': int(offer_term)}

print("✅ Математические функции готовы к расчету!")

✅ Математические функции готовы к расчету!


###  Генерация 10 эталонных персон

* **Кейсы 1–3 (`APPROVED`):** надежные компании из разных отраслей с комфортным графиком платежей.
* **Кейсы 4–7 (`APPROVED_WITH_CHANGES`):** компании, попадающие в Желтую зону (высокий риск кассового разрыва $P_{\text{gap}} \ge 25\%$ или повышенный $PD$), которым система автоматически реструктурирует сумму и срок.
* **Кейсы 8–10 (`REJECTED`):** компании в Красной зоне (наличие просрочек в БКИ, критическая долговая нагрузка или недостаток денежного потока).

In [15]:
personas = [
    # 1. Зеленая зона: Зрелая IT-компания, низкий долг, хорошая подушка C_0
    {'case_id': 1, 'inn': '7701000001', 'product_code': 'WORKING_CAP', 'amount': 2000000.0, 'term_months': 12, 'monthly_revenue': 3000000.0, 'd_old': 150000.0, 'c_0': 600000.0, 'business_age_months': 48, 'business_type': 'ooo', 'has_delinquency': 0, 'okved_group': 'it'},

    # 2. Зеленая зона: Надежное производство, покупка оборудования на 36 мес
    {'case_id': 2, 'inn': '7701000002', 'product_code': 'EQUIPMENT', 'amount': 5000000.0, 'term_months': 36, 'monthly_revenue': 4000000.0, 'd_old': 200000.0, 'c_0': 900000.0, 'business_age_months': 60, 'business_type': 'ooo', 'has_delinquency': 0, 'okved_group': 'manufacturing'},

    # 3. Зеленая зона: Опытный ИП в ритейле, небольшой кредит на кассовый разрыв
    {'case_id': 3, 'inn': '7701000003', 'product_code': 'CASH_GAP', 'amount': 800000.0, 'term_months': 6, 'monthly_revenue': 2500000.0, 'd_old': 50000.0, 'c_0': 300000.0, 'business_age_months': 42, 'business_type': 'ip', 'has_delinquency': 0, 'okved_group': 'retail'},

    # 4. Желтая зона (по P_gap): Отличная IT-компания, но просит слишком много на 12 мес -> увеличиваем срок до 24 мес
    {'case_id': 4, 'inn': '7701000004', 'product_code': 'EQUIPMENT', 'amount': 6000000.0, 'term_months': 12, 'monthly_revenue': 2000000.0, 'd_old': 150000.0, 'c_0': 120000.0, 'business_age_months': 48, 'business_type': 'ooo', 'has_delinquency': 0, 'okved_group': 'it'},

    # 5. Желтая зона (по P_gap + урезание суммы): Строительная компания просит 8 млн на 12 мес -> снижаем сумму и увеличиваем срок
    {'case_id': 5, 'inn': '7701000005', 'product_code': 'WORKING_CAP', 'amount': 8000000.0, 'term_months': 12, 'monthly_revenue': 3000000.0, 'd_old': 150000.0, 'c_0': 240000.0, 'business_age_months': 40, 'business_type': 'ooo', 'has_delinquency': 0, 'okved_group': 'construction'},

    # 6. Желтая зона (по PD между 5% и 15%): Молодое ООО в HoReCa со средней нагрузкой -> реструктуризация
    {'case_id': 6, 'inn': '7701000006', 'product_code': 'WORKING_CAP', 'amount': 1500000.0, 'term_months': 12, 'monthly_revenue': 1500000.0, 'd_old': 120000.0, 'c_0': 180000.0, 'business_age_months': 18, 'business_type': 'ooo', 'has_delinquency': 0, 'okved_group': 'horeca'},

    # 7. Желтая зона: ИП со средней долговой нагрузкой (45%) и стажем > 3 лет
    {'case_id': 7, 'inn': '7701000007', 'product_code': 'EQUIPMENT', 'amount': 2500000.0, 'term_months': 24, 'monthly_revenue': 2000000.0, 'd_old': 225000.0, 'c_0': 240000.0, 'business_age_months': 50, 'business_type': 'ip', 'has_delinquency': 0, 'okved_group': 'manufacturing'},

    # 8. Красная зона (REJECTED по PD >= 15%): Высокая долговая нагрузка (>60%) + просрочки в БКИ
    {'case_id': 8, 'inn': '7701000008', 'product_code': 'WORKING_CAP', 'amount': 3000000.0, 'term_months': 24, 'monthly_revenue': 1500000.0, 'd_old': 160000.0, 'c_0': 50000.0, 'business_age_months': 10, 'business_type': 'ip', 'has_delinquency': 1, 'okved_group': 'retail'},

    # 9. Красная зона (REJECTED по PD >= 15%): Молодой бизнес с просрочками и высокой нагрузкой
    {'case_id': 9, 'inn': '7701000009', 'product_code': 'CASH_GAP', 'amount': 1000000.0, 'term_months': 6, 'monthly_revenue': 1000000.0, 'd_old': 100000.0, 'c_0': 20000.0, 'business_age_months': 6, 'business_type': 'ooo', 'has_delinquency': 1, 'okved_group': 'horeca'},

    # 10. Красная зона (REJECTED из-за нехватки свободного потока даже после реструктуризации)
    {'case_id': 10, 'inn': '7701000010', 'product_code': 'WORKING_CAP', 'amount': 2000000.0, 'term_months': 12, 'monthly_revenue': 400000.0, 'd_old': 55000.0, 'c_0': 0.0, 'business_age_months': 40, 'business_type': 'ip', 'has_delinquency': 0, 'okved_group': 'retail'}
]

# --- ЗАДАЧА 5: Сохраняем чистую таблицу персон (personas.csv) ---
persona_cols = ['case_id', 'inn', 'business_type', 'okved_group', 'business_age_months', 'monthly_revenue', 'd_old', 'c_0', 'has_delinquency']
df_personas = pd.DataFrame(personas)[persona_cols]
df_personas.to_csv('personas.csv', index=False)

print("=== ТАБЛИЦА ПЕРСОН PERSONAS.CSV (ЗАДАЧА 5) ===")
display(df_personas)

# --- ЗАДАЧА 6: Прогоняем все 10 персон через конвейер (golden_cases.csv) ---
df_golden = pd.DataFrame([solve_persona(p) for p in personas])

cols_order = [
    'case_id', 'inn', 'product_code', 'amount', 'term_months',
    'monthly_revenue', 'd_old', 'c_0', 'business_age_months', 'debt_burden_ratio', 'business_type', 'has_delinquency', 'okved_group',
    'score', 'pd_client', 'p_gap', 'rate', 'payment', 'status', 'offer_amount', 'offer_term'
]
df_golden = df_golden[cols_order]

df_golden.to_csv('golden_cases.csv', index=False)

print("\n=== ЭТАЛОННАЯ ТАБЛИЦА GOLDEN_CASES.CSV (ЗАДАЧА 6) ===")
display(df_golden)

=== ТАБЛИЦА ПЕРСОН PERSONAS.CSV (ЗАДАЧА 5) ===


,case_id,inn,business_type,okved_group,business_age_months,monthly_revenue,d_old,c_0,has_delinquency
0,1,7701000001,ooo,it,48,"3,000,000.0000","150,000.0000","600,000.0000",0
1,2,7701000002,ooo,manufacturing,60,"4,000,000.0000","200,000.0000","900,000.0000",0
2,3,7701000003,ip,retail,42,"2,500,000.0000","50,000.0000","300,000.0000",0
3,4,7701000004,ooo,it,48,"2,000,000.0000","150,000.0000","120,000.0000",0
4,5,7701000005,ooo,construction,40,"3,000,000.0000","150,000.0000","240,000.0000",0
5,6,7701000006,ooo,horeca,18,"1,500,000.0000","120,000.0000","180,000.0000",0
6,7,7701000007,ip,manufacturing,50,"2,000,000.0000","225,000.0000","240,000.0000",0
7,8,7701000008,ip,retail,10,"1,500,000.0000","160,000.0000","50,000.0000",1
8,9,7701000009,ooo,horeca,6,"1,000,000.0000","100,000.0000","20,000.0000",1
9,10,7701000010,ip,retail,40,"400,000.0000","55,000.0000",0.0000,0



=== ЭТАЛОННАЯ ТАБЛИЦА GOLDEN_CASES.CSV (ЗАДАЧА 6) ===


,case_id,inn,product_code,amount,term_months,monthly_revenue,d_old,c_0,business_age_months,debt_burden_ratio,...,has_delinquency,okved_group,score,pd_client,p_gap,rate,payment,status,offer_amount,offer_term
0,1,7701000001,WORKING_CAP,"2,000,000.0000",12,"3,000,000.0000","150,000.0000","600,000.0000",48,0.1429,...,0,it,666,0.0020,0.0000,0.1924,"184,542.3200",APPROVED,"2,000,000.0000",12
1,2,7701000002,EQUIPMENT,"5,000,000.0000",36,"4,000,000.0000","200,000.0000","900,000.0000",60,0.2000,...,0,manufacturing,666,0.0020,0.0000,0.1824,"181,364.5300",APPROVED,"5,000,000.0000",36
2,3,7701000003,CASH_GAP,"800,000.0000",6,"2,500,000.0000","50,000.0000","300,000.0000",42,0.1333,...,0,retail,649,0.0036,0.0000,0.2043,"141,390.0800",APPROVED,"800,000.0000",6
3,4,7701000004,EQUIPMENT,"6,000,000.0000",12,"2,000,000.0000","150,000.0000","120,000.0000",48,0.2143,...,0,it,666,0.0020,0.4650,0.1824,"300,240.8500",APPROVED_WITH_CHANGES,"6,000,000.0000",24
4,5,7701000005,WORKING_CAP,"8,000,000.0000",12,"3,000,000.0000","150,000.0000","240,000.0000",40,0.2500,...,0,construction,666,0.0020,0.9631,0.1924,"321,846.1500",APPROVED_WITH_CHANGES,"6,370,000.0000",24
5,6,7701000006,WORKING_CAP,"1,500,000.0000",12,"1,500,000.0000","120,000.0000","180,000.0000",18,0.4444,...,0,horeca,591,0.0266,0.3404,0.2225,"78,002.5400",APPROVED_WITH_CHANGES,"1,500,000.0000",24
6,7,7701000007,EQUIPMENT,"2,500,000.0000",24,"2,000,000.0000","225,000.0000","240,000.0000",50,0.4500,...,0,manufacturing,614,0.0122,0.0041,0.1946,"126,580.9800",APPROVED,"2,500,000.0000",24
7,8,7701000008,WORKING_CAP,"3,000,000.0000",24,"1,500,000.0000","160,000.0000","50,000.0000",10,0.7111,...,1,retail,445,0.8115,1.0000,0.0000,0.0000,REJECTED,0.0000,0
8,9,7701000009,CASH_GAP,"1,000,000.0000",6,"1,000,000.0000","100,000.0000","20,000.0000",6,0.5556,...,1,horeca,504,0.3578,1.0000,0.0000,0.0000,REJECTED,0.0000,0
9,10,7701000010,WORKING_CAP,"2,000,000.0000",12,"400,000.0000","55,000.0000",0.0000,40,0.9167,...,0,retail,572,0.0501,1.0000,0.0000,0.0000,REJECTED,0.0000,0


###  Задача 5: Матрица корреляций признаков ($R$) и проверка мультиколлинеарности
Для подтверждения корректности отбора факторов в скоринговую карту (`scorecard_bins.csv`) и реалистичности сформированных персон рассчитываем матрицу парных корреляций Пирсона $R$:
1. **Экономическая логика:** возраст бизнеса положительно коррелирует с выручкой и размером финансовой подушки $C_0$, и отрицательно — с вероятностью дефолта `pd_client`.
2. **Контроль мультиколлинеарности факторов карты:** ни одна пара входных факторов скоринговой карты не должна превышать порог сильной коллинеарности $|r_{ij}| < 0.70$, чтобы избежать дублирования баллов.

In [4]:
# Создаем копию таблицы для расчета корреляций и переводим 'ooo'/'ip' в 1/0
df_for_corr = df_golden.copy()
df_for_corr['is_ooo'] = (df_for_corr['business_type'] == 'ooo').astype(int)

# 1. Матрица корреляций R по 4 входным факторам скоринговой карты (проверка |r| < 0.70)
scorecard_factors = ['business_age_months', 'debt_burden_ratio', 'has_delinquency', 'is_ooo']
df_corr_scorecard = df_for_corr[scorecard_factors].corr().round(2)

print("=== 1. МАТРИЦА КОРРЕЛЯЦИЙ R (Факторы скоринговой карты) ===")
display(df_corr_scorecard)

# Автоматическая проверка максимальной корреляции вне главной диагонали
max_corr = df_corr_scorecard.where(~pd.eye(len(df_corr_scorecard), dtype=bool)).abs().max().max()
print(f"✅ Максимальная корреляция между факторами карты: {max_corr:.2f} (норма < 0.70 — мультиколлинеарность отсутствует)\n")

# 2. Полная матрица корреляций R по всем параметрам персон и итоговому PD
all_metrics = ['business_age_months', 'monthly_revenue', 'c_0', 'debt_burden_ratio', 'has_delinquency', 'is_ooo', 'pd_client']
df_corr_full = df_for_corr[all_metrics].corr().round(2)
df_corr_full.to_csv('correlation_matrix_R.csv')

print("=== 2. ПОЛНАЯ МАТРИЦА КОРРЕЛЯЦИЙ R ПО ПЕРСОНАМ (сохранена в correlation_matrix_R.csv) ===")
display(df_corr_full)

=== 1. МАТРИЦА КОРРЕЛЯЦИЙ R (Факторы скоринговой карты) ===


,business_age_months,debt_burden_ratio,has_delinquency,is_ooo
business_age_months,1.0000,-0.5300,-0.8100,0.0300
debt_burden_ratio,-0.5300,1.0000,0.4600,-0.4900
has_delinquency,-0.8100,0.4600,1.0000,-0.1000
is_ooo,0.0300,-0.4900,-0.1000,1.0000


AttributeError: module 'pandas' has no attribute 'eye'